# Topology Visualization

This notebook visualizes different network topologies including intra-node structures.

## Hierarchy

The topology supports a multi-level hierarchy:
- **Server**: Generated by the inter-network topology, connects directly to topology switch (ToR)
- **Nodes per Server**: Each server can have multiple nodes (controlled by `nodes_per_server`)
- **NPUs per Node**: Each node has multiple NPUs connected via intra-node topology (switch/ring/fully_connected)

For **Dragonfly/Jellyfish**: All switches ARE ToRs  
For **FoldedClos**: Only edge switches are ToRs; aggregation and core switches form the upper layers

**Structure**: NPUs -> NVSwitch/NPU-Switch -> NIC (ring/fc only) -> Edge/Switch (ToR) -> [Agg -> Core] (FoldedClos only)

## Visualization Features (bottom to top)
- **NPUs** (green): Processing units at level 0
- **NVSwitches/NPU-switches** (orange): Intra-node switches at level 1
- **NIC switches** (pink): For ring/fully_connected, connects node to switch at level 2
- **Edge switches / ToR** (blue, FoldedClos) or **Switches** (purple, Dragonfly/Jellyfish): at level 3
- **Aggregation switches** (purple, FoldedClos only): at level 4
- **Core switches** (red, FoldedClos only): at level 5
- Edge widths proportional to bandwidth

In [ ]:
import sys
sys.path.insert(0, '/app/astra-sim/optimos/create_topology/new')

import importlib
import matplotlib.pyplot as plt

# Reload module to pick up any changes
import create_topology
importlib.reload(create_topology)
from create_topology import CustomizedDragonfly, FoldedClos, Jellyfish
import utils
importlib.reload(utils)
from utils import visualize_topology, visualize_single_node, visualize_single_server, build_topology

## Example 1: Small Dragonfly with NVSwitch Intra-node

4 groups, 4 switches per group (all switches ARE ToRs), 2 servers per switch, 4 NPUs per node, 1 NVSwitch

In [ ]:
# Small dragonfly with NVSwitch topology
# All dragonfly switches serve as ToRs (no separate ToR level)
bw_config = {
    'host_switch': 100,       # Inter-node bandwidth (NVSwitch to switch/ToR)
    'intra_group': 200,       # Between switches in same group
    'inter_group': 50,        # Between groups
    'intra_node': 400         # Within node (NPU to NVSwitch)
}

latency_config = {
    'host_switch': 0.001,     # Inter-node latency
    'intra_group': 0.01,      # Between switches in same group
    'inter_group': 0.1,       # Between groups
    'intra_node': 0.0001      # Within node (NPU to NVSwitch)
}

df_nvswitch = CustomizedDragonfly(
    G=4,                      # 4 groups
    A=4,                      # 4 switches per group (all are ToRs)
    h=2,                      # 2 inter-pod links per switch
    concentration=2,          # 2 servers per switch
    bandwidth_config=bw_config,
    latency_config=latency_config,
    npus_per_node=4,          # 4 NPUs per node
    intra_node_topology='switch',
    num_nvswitches=1
)

fig, ax, G = visualize_topology(df_nvswitch, "Dragonfly with NVSwitch (switches ARE ToRs)")
plt.show()

print(f"Total nodes: {G.number_of_nodes()}")
print(f"Total edges: {G.number_of_edges()}")

## Example 2: Dragonfly with Multiple NVSwitches

In [ ]:
# Dragonfly with NVSwitch - larger example
bw_config = {
    'host_switch': 100,
    'intra_group': 200,
    'inter_group': 50,
    'intra_node': 400
}

latency_config = {
    'host_switch': 0.001,
    'intra_group': 0.01,
    'inter_group': 0.1,
    'intra_node': 0.0001
}

df_multi_nv = CustomizedDragonfly(
    G=2,
    A=2,
    h=1,
    concentration=10,
    bandwidth_config=bw_config,
    latency_config=latency_config,
    npus_per_node=2,
    intra_node_topology='switch',
    num_nvswitches=1
)

fig, ax, G = visualize_topology(df_multi_nv, "Dragonfly - Switches are ToRs (no separate ToR)")
plt.show()

## Example 3: Ring Intra-node Topology

In [ ]:
# Dragonfly with Ring intra-node topology
bw_config = {
    'host_switch': 100,
    'intra_group': 200,
    'inter_group': 50,
    'intra_node': 400
}

latency_config = {
    'host_switch': 0.001,
    'intra_group': 0.01,
    'inter_group': 0.1,
    'intra_node': 0.0001
}

df_ring = CustomizedDragonfly(
    G=2,
    A=1,                       # Smaller for clarity
    h=1,
    concentration=2,
    nodes_per_server=2,
    bandwidth_config=bw_config,
    latency_config=latency_config,
    npus_per_node=2,
    intra_node_topology='ring'
)

fig, ax, G = visualize_topology(df_ring, "Dragonfly with Ring Intra-node Topology")
plt.show()

## Example 4: Fully Connected Intra-node Topology

In [ ]:
# Dragonfly with Fully Connected intra-node topology
bw_config = {
    'host_switch': 100,
    'intra_group': 200,
    'inter_group': 50,
    'intra_node': 400
}

latency_config = {
    'host_switch': 0.001,
    'intra_group': 0.01,
    'inter_group': 0.1,
    'intra_node': 0.0001
}

df_fc = CustomizedDragonfly(
    G=2,
    A=1,
    h=1,
    concentration=2,
    bandwidth_config=bw_config,
    latency_config=latency_config,
    nodes_per_server=1,
    npus_per_node=4,
    intra_node_topology='fully_connected'
)

fig, ax, G = visualize_topology(df_fc, "Dragonfly with Fully Connected Intra-node")
plt.show()

## Example 5: Single Node Detail View

Zoom in on just one node to see the NVSwitch -> Switch (ToR) structure

In [ ]:
# Visualize single node with 1 NVSwitch
# The topology switch itself IS the ToR
bw_config = {
    'host_switch': 100,
    'intra_group': 200,
    'inter_group': 50,
    'intra_node': 400
}

latency_config = {
    'host_switch': 0.001,
    'intra_group': 0.01,
    'inter_group': 0.1,
    'intra_node': 0.0001
}

df_single = CustomizedDragonfly(
    G=2, A=1, h=1, concentration=1,
    bandwidth_config=bw_config,
    latency_config=latency_config,
    npus_per_node=4,
    intra_node_topology='switch',
    num_nvswitches=1
)

visualize_single_node(df_single, node_idx=1)
plt.show()

print("\nTopology structure:")
print(f"NPUs: {df_single.node_npus}")
print(f"NVSwitches: {df_single.nvswitch_ids}")
print(f"Switches serving as ToRs: {df_single.tor_switches}")

In [ ]:
# Visualize single node with 2 NVSwitches
bw_config = {
    'host_switch': 100,
    'intra_group': 200,
    'inter_group': 50,
    'intra_node': 400
}

latency_config = {
    'host_switch': 0.001,
    'intra_group': 0.01,
    'inter_group': 0.1,
    'intra_node': 0.0001
}

df_single_2nv = CustomizedDragonfly(
    G=1, A=1, h=1, concentration=1,
    bandwidth_config=bw_config,
    latency_config=latency_config,
    npus_per_node=4,
    intra_node_topology='switch',
    num_nvswitches=2
)

visualize_single_node(df_single_2nv, node_idx=0)
plt.show()

print("\nWith 2 NVSwitches - note bandwidth is split:")
print(f"NVSwitches: {df_single_2nv.nvswitch_ids}")

## Verify the New Structure

Check that NVSwitches connect directly to the main switch (which IS the ToR) - no separate ToR level

In [ ]:
# Print detailed link information
bw_config = {
    'host_switch': 100,
    'intra_group': 200,
    'inter_group': 50,
    'intra_node': 400
}

latency_config = {
    'host_switch': 0.001,
    'intra_group': 0.01,
    'inter_group': 0.1,
    'intra_node': 0.0001
}

topo = CustomizedDragonfly(
    G=1, A=1, h=1, concentration=1,
    bandwidth_config=bw_config,
    latency_config=latency_config,
    npus_per_node=4,
    intra_node_topology='switch',
    num_nvswitches=1
)
build_topology(topo)

print("Links in the topology:")
print("="*70)
print(f"{'Link ID':<10} {'Source':<10} {'Dest':<10} {'Bandwidth':<12} {'Latency':<10}")
print("-"*70)

for link_id, (src, dst) in sorted(topo.links.items()):
    bw = topo.link_bandwidths.get(link_id, 'N/A')
    lat = topo.link_latencies.get(link_id, 'N/A')
    print(f"{link_id:<10} {src:<10} {dst:<10} {bw:<12} {lat:<10}")

print("\n" + "="*70)
print("Verification:")
print(f"- Main switches (ToRs): {topo.tor_switches}")
print(f"- NVSwitches: {topo.nvswitch_ids}")
print(f"\nConnections to main switch t1 (NVSwitches should connect directly):")
for link_id, (src, dst) in topo.links.items():
    if dst == 't1' or src == 't1':
        lat = topo.link_latencies.get(link_id, 'N/A')
        print(f"  {src} <-> {dst} (bw: {topo.link_bandwidths.get(link_id)}, lat: {lat})")

## Example 6: Folded-Clos (Fat-Tree) Topology

K=4 fat-tree with NVSwitch intra-node topology.
- **Core switches** (red, top): K²/4 = 4 core switches
- **Aggregation switches** (purple, middle): K/2 per pod = 8 total
- **Edge switches** (blue, bottom): K/2 per pod = 8 total (these ARE the ToRs)

In [ ]:
# Folded-Clos (Fat-Tree) with NVSwitch topology
# 3 switch layers: Core (top) -> Aggregation -> Edge/ToR (bottom)
bw_config = {
    'host_edge': 100,         # NVSwitch to edge switch (ToR)
    'edge_agg': 200,          # Edge to aggregation
    'agg_core': 200,          # Aggregation to core
    'intra_node': 400         # Within node (NPU to NVSwitch)
}

latency_config = {
    'host_edge': 0.001,       # NVSwitch to edge switch latency
    'edge_agg': 0.01,         # Edge to aggregation latency
    'agg_core': 0.05,         # Aggregation to core latency
    'intra_node': 0.0001      # Within node latency
}

fc_nvswitch = FoldedClos(
    K=4,                      # K=4 fat-tree
    bandwidth_config=bw_config,
    latency_config=latency_config,
    npus_per_node=2,          # 2 NPUs per node (smaller for visibility)
    intra_node_topology='switch',
    num_nvswitches=1
)

fig, ax, G = visualize_topology(fc_nvswitch, "Folded-Clos (K=4): Core -> Agg -> Edge(ToR)", figsize=(16, 12), show_bw=False)
plt.show()

print(f"Total graph nodes: {G.number_of_nodes()}")
print(f"Total graph edges: {G.number_of_edges()}")
print(f"NPUs: {len(fc_nvswitch.all_npus)}")
print(f"Edge switches (ToRs): {fc_nvswitch.edge_switches}")
print(f"Aggregation switches: {fc_nvswitch.agg_switches}")
print(f"Core switches: {fc_nvswitch.core_switches}")

## Example 7: Jellyfish Topology

Random graph topology with intra-node support. All Jellyfish switches ARE ToRs.

In [ ]:
# Jellyfish with fully_connected intra-node topology
# All Jellyfish switches serve as ToRs
bw_config = {
    'host_switch': 100,       # NIC to switch (ToR)
    'switch_switch': 200,     # Between switches
    'intra_node': 400         # Within node (NPU to NPU-switch)
}

latency_config = {
    'host_switch': 0.001,     # NIC to switch latency
    'switch_switch': 0.01,    # Between switches latency
    'intra_node': 0.0001      # Within node latency
}

jf_nvswitch = Jellyfish(
    num_switches=6,           # 6 switches (all are ToRs)
    degree=3,                 # Each switch connects to 3 others
    num_hosts_per_switch=1,   # 1 server per switch
    bandwidth_config=bw_config,
    latency_config=latency_config,
    nodes_per_server=2,
    npus_per_node=2,
    intra_node_topology='fully_connected',
)

fig, ax, G = visualize_topology(jf_nvswitch, "Jellyfish - Switches ARE ToRs", figsize=(14, 10))
plt.show()

print(f"Total nodes: {G.number_of_nodes()}")
print(f"Total edges: {G.number_of_edges()}")
print(f"NPUs: {len(jf_nvswitch.all_npus)}")
print(f"Switches (ToRs): {jf_nvswitch.tor_switches}")

## Example 8: Single Node View - Folded-Clos

Visualize just one node from a FoldedClos topology to see the NVSwitch -> Edge Switch (ToR) structure.
The edge switch is shown in **blue** to distinguish it from aggregation/core switches.

In [ ]:
# Single node view from Folded-Clos with 2 NVSwitches
# Edge switch (ToR) is shown in blue - distinct from agg/core switches
bw_config = {
    'host_edge': 100,
    'edge_agg': 200,
    'agg_core': 200,
    'intra_node': 600  # High intra-node bandwidth
}

latency_config = {
    'host_edge': 0.001,
    'edge_agg': 0.01,
    'agg_core': 0.05,
    'intra_node': 0.00005  # Low intra-node latency
}

fc_single = FoldedClos(
    K=4,
    bandwidth_config=bw_config,
    latency_config=latency_config,
    npus_per_node=4,
    intra_node_topology='switch',
    num_nvswitches=2  # 2 NVSwitches
)

visualize_single_node(fc_single, node_idx=1)
plt.show()

print("\nNode structure with 2 NVSwitches:")
print(f"NVSwitches: {fc_single.nvswitch_ids[:4]}...")
print(f"Edge switches (ToRs): {fc_single.edge_switches}")
print(f"Intra-node BW per NVSwitch: {bw_config['intra_node'] / 2}")

## Example 9: Multi-Node Server (Server -> Nodes -> NPUs)

This example demonstrates the `nodes_per_server` parameter which creates a multi-node hierarchy:
- **Server**: connects directly to the topology switch (ToR)
- **Nodes**: multiple nodes per server, each with its own NVSwitch(es)
- **NPUs**: multiple NPUs per node

Structure: NPUs -> NVSwitch -> Switch (ToR)

In [ ]:
# Multi-Node Server: Server -> Nodes -> NPUs
# Topology switches serve directly as ToRs
bw_config = {
    'host_switch': 100,       # Inter-node bandwidth (NVSwitch to switch/ToR)
    'intra_group': 200,       # Between switches in same group
    'inter_group': 50,        # Between groups
    'intra_node': 400         # Within node (NPU to NVSwitch)
}

latency_config = {
    'host_switch': 0.001,
    'intra_group': 0.01,
    'inter_group': 0.1,
    'intra_node': 0.0001
}

df_3level = CustomizedDragonfly(
    G=2,                      # 2 groups
    A=2,                      # 2 switches per group (all are ToRs)
    h=1,                      # 1 inter-group link per switch
    concentration=1,          # 1 server per switch
    bandwidth_config=bw_config,
    latency_config=latency_config,
    nodes_per_server=2,       # 2 nodes per server
    npus_per_node=4,          # 4 NPUs per node
    intra_node_topology='switch',
    num_nvswitches=1
)

fig, ax, G = visualize_topology(df_3level, 
    "Multi-Node: Server (1) -> Nodes (2) -> NPUs (4), Switches=ToRs", 
    figsize=(16, 12), show_bw=False)
plt.show()

print(f"Configuration:")
print(f"  - Switches (ToRs): {df_3level.total_num_switches}")
print(f"  - Servers (total): {df_3level.total_num_servers}")
print(f"  - Nodes per server: {df_3level.nodes_per_server}")
print(f"  - Total nodes: {df_3level.total_num_nodes}")
print(f"  - NPUs per node: {df_3level.npus_per_node}")
print(f"  - Total NPUs: {df_3level.total_num_hosts}")
print(f"\nHierarchy mapping:")
print(f"  - server_nodes: {dict(list(df_3level.server_nodes.items())[:2])}...")
print(f"  - node_npus: {dict(list(df_3level.node_npus.items())[:2])}...")

## Example 10: Single Server View (Multi-Node)

Visualize a single server with multiple nodes to see the full hierarchy:
NPUs -> NVSwitch(es) -> Switch (ToR)

In [ ]:
# Visualize server 1 from the 3-level hierarchy topology
visualize_single_server(df_3level, server_idx=1)
plt.show()

print(f"\nServer 1 contains:")
print(f"  - Nodes: {df_3level.server_nodes.get(1, [])}")
for node_id in df_3level.server_nodes.get(1, []):
    print(f"  - Node {node_id} NPUs: {df_3level.node_npus.get(node_id, [])}")

## Example 11: Folded-Clos with Multi-Node Servers

Fat-Tree topology with multiple nodes per server.
- **Edge switches** (blue) ARE the ToRs - servers connect directly to them
- Single-server view shows **Edge Switch (ToR)** in blue at the top

In [ ]:
# Folded-Clos with 3-Level Hierarchy
# Edge switches (blue) ARE the ToRs
bw_config = {
    'host_edge': 100,         # Host to edge switch (ToR)
    'edge_agg': 200,          # Edge to aggregation
    'agg_core': 200,          # Aggregation to core
    'intra_node': 400         # Within node (NPU to NVSwitch)
}

latency_config = {
    'host_edge': 0.001,
    'edge_agg': 0.01,
    'agg_core': 0.05,
    'intra_node': 0.0001
}

fc_3level = FoldedClos(
    K=4,                      # K=4 fat-tree
    bandwidth_config=bw_config,
    latency_config=latency_config,
    nodes_per_server=2,       # 2 nodes per server
    npus_per_node=8,          # 8 NPUs per node
    intra_node_topology='switch',
    num_nvswitches=1
)

print(f"Folded-Clos Configuration:")
print(f"  - K: {fc_3level.K}")
print(f"  - Total servers: {fc_3level.total_num_servers}")
print(f"  - Nodes per server: {fc_3level.nodes_per_server}")
print(f"  - Total nodes: {fc_3level.total_num_nodes}")
print(f"  - NPUs per node: {fc_3level.npus_per_node}")
print(f"  - Total NPUs: {fc_3level.total_num_hosts}")
print(f"  - Edge switches (ToRs): {fc_3level.edge_switches}")
print(f"  - Aggregation switches: {fc_3level.agg_switches}")
print(f"  - Core switches: {fc_3level.core_switches}")

# Visualize a single server from the FoldedClos
# Edge switch (ToR) will appear in blue
visualize_single_server(fc_3level, server_idx=1)
plt.show()

## Example 12: Dragonfly with One NPU per Switch

This example creates a Dragonfly topology where each switch is connected to exactly one NPU.
This is achieved by setting:
- `concentration=1` (1 server per switch)
- `nodes_per_server=1` (1 node per server)
- `npus_per_node=1` (1 NPU per node)

In [ ]:
# Dragonfly with 1 NPU per switch
bw_config = {
    'host_switch': 100,
    'intra_group': 200,
    'inter_group': 50,
    'intra_node': 1 # Not relevant with 1 NPU
}

latency_config = {
    'host_switch': 0.001,
    'intra_group': 0.01,
    'inter_group': 0.1,
    'intra_node': 0.0001
}

df_one_npu = CustomizedDragonfly(
    G=2,
    A=2,
    h=1,
    concentration=1,          # 1 server per switch
    nodes_per_server=1,       # 1 node per server
    bandwidth_config=bw_config,
    latency_config=latency_config,
    npus_per_node=1,          # 1 NPU per node
    intra_node_topology='switch' # Not relevant with 1 NPU
)

fig, ax, G = visualize_topology(df_one_npu, "Dragonfly with 1 NPU per Switch")
plt.show()

print(f"Total NPUs: {df_one_npu.total_num_hosts}")
print(f"Total Switches: {df_one_npu.total_num_switches}")